# 06 — Risk-controlled momentum grid

Purpose: take the current best research signal, historical 10D momentum, and test whether simple volatility controls can improve ICIR and reduce drawdown before adding more complex models.


In [ ]:
import sys, json, warnings
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init
from src.research.notebook_lab_contracts import ResearchSessionConfig, CANONICAL_10D_RETURN_EXPR
from src.research.notebook_experiment_api import run_10d_experiment
from src.research.risk_controlled_momentum import build_risk_controlled_momentum_grid
from src.research.ten_day_model_gates import summarize_report_gates

cfg_path = ROOT / "data" / "session_config.json"
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding="utf-8"))
    MARKET, SYMBOLS, BENCHMARK = cfg["market"], cfg["symbols"], cfg["benchmark"]
    TRAIN_START, TRAIN_END = cfg["train_start"], cfg["train_end"]
    TEST_START, TEST_END = cfg["test_start"], cfg["test_end"]
else:
    MARKET = "us"
    SYMBOLS = ["AAPL", "NVDA", "MSFT", "GOOGL", "AMZN", "META", "TSLA", "AVGO", "COST", "NFLX"]
    BENCHMARK = "QQQ"
    TRAIN_START, TRAIN_END = "2021-01-01", "2024-12-31"
    TEST_START, TEST_END = "2025-01-01", "2026-06-18"

config = ResearchSessionConfig(
    market=MARKET,
    symbols=SYMBOLS,
    benchmark=BENCHMARK,
    train_start=TRAIN_START,
    train_end=TRAIN_END,
    test_start=TEST_START,
    test_end=TEST_END,
    topk=15,
    experiment_id=f"{MARKET}_risk_controlled_momentum_grid",
    return_expression=CANONICAL_10D_RETURN_EXPR,
)

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D

print(config)


In [ ]:
momentum = D.features(SYMBOLS, ["$close/Ref($close,10)-1"], start_time=TEST_START, end_time=TEST_END)
volatility = D.features(SYMBOLS, ["Std($ret,10)"], start_time=TEST_START, end_time=TEST_END)
raw = D.features(SYMBOLS, [config.return_expression], start_time=TEST_START, end_time=TEST_END)

for frame in (momentum, volatility, raw):
    if frame.index.names == ["instrument", "datetime"]:
        frame.index = frame.index.swaplevel()
        frame.sort_index(inplace=True)

momentum.columns = ["score"]
volatility.columns = ["volatility"]
raw_returns = raw.copy()
raw_returns.columns = ["return"]
raw_returns.attrs["provenance"] = "raw_forward_return"
raw_returns.attrs["horizon"] = 10
raw_returns.attrs["expression"] = config.return_expression

print(momentum.shape, volatility.shape, raw_returns.shape)


In [ ]:
candidates = {"factor:historical_momentum_10d": momentum}
candidates.update(
    build_risk_controlled_momentum_grid(
        momentum,
        volatility,
        volatility_quantiles=(0.50, 0.60, 0.70, 0.80, 0.90),
    )
)

experiment = run_10d_experiment(
    config=config,
    candidates=candidates,
    raw_returns=raw_returns,
    output_dir=ROOT / "artifacts" / "evidence" / "notebook_10d_lab",
)

report = experiment["comparison_report"]
print(json.dumps(report["summary"], indent=2, default=str))
print(json.dumps(summarize_report_gates(report), indent=2, default=str))
